# solvekit.core

> Core utilities for extending the solveit UI

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
import time
from textwrap import dedent
from typing import Literal

import httpx
from dialoghelper.core import iife

In [ ]:
#| export
def _ensure_toolbar():
    """Register `window.solvekit.addBtn` JS helper in the browser, enabling custom toolbar buttons in the solveit toolbar."""
    iife(dedent("""
        window.solvekit = window.solvekit || {};
        if (typeof window.solvekit.addBtn === 'undefined') {
            window.solvekit.addBtn = function(id, icon, onclick, tooltip=null) {
                const toolbar = document.querySelector('nav.monster-navbar .flex.justify-end');
                if (!toolbar) return console.warn('toolbar not found');

                let btn_group = document.getElementById('custom_group')
                if (!btn_group) {
                    btn_group = document.createElement('div');
                    btn_group.className = 'uk-btn-group';
                    btn_group.id = 'custom_group';
                    toolbar.prepend(btn_group);
                }

                let btn = document.getElementById(id);
                if (!btn) {
                    btn = document.createElement('button');
                    btn.id = id;
                    btn.className = 'uk-btn uk-btn-icon uk-btn-sm text-lg uk-btn-default cursor-pointer';
                    btn_group.appendChild(btn);
                }
                
                if (tooltip) 
                    btn.setAttribute('uk-tooltip', tooltip);
                else 
                    btn.removeAttribute('uk-tooltip')
                btn.innerHTML = icon;
                btn.onclick = onclick;
            }
        }""").strip())

**SVG Icon Support**: The `icon` parameter supports the `'ii:{prefix}:{name}'` format, which automatically fetches SVG icons from the [Iconify API](https://api.iconify.design). For example, `'ii:lucide:copy-plus'` will request https://api.iconify.design/lucide/copy-plus.svg?height=16&width=16, with a default size of 16×16.

In [ ]:
#| export
def _resolve_icon(
    icon:str, # Icon string, e.g. 'ii:lucide:copy-plus' or raw SVG/HTML
    size:int=16 # Icon dimensions in pixels (width and height)
):
    "Resolve an `ii:{prefix}:{name}` icon string to SVG via the Iconify API, or return as-is."
    if not icon.startswith('ii:'): return icon
    _, prefix, name = icon.split(':', 2)
    try:
        r = httpx.get(f'https://api.iconify.design/{prefix}/{name}.svg?height={size}&width={size}', timeout=3)
        r.raise_for_status()
        return r.text
    except httpx.HTTPError as e:
        raise ValueError(f'Failed to fetch icon "{prefix}:{name}"') from e

In [ ]:
#| export
def add_btn(
    id:str,   # Unique DOM id for the button element
    icon:str,     # HTML/SVG icon string, or `'ii:prefix:name'` for Iconify lookup
    onclick:str,  # Code to execute on click (Python or JS depending on `lang`)
    tooltip:str=None,  # Optional tooltip text shown on hover
    lang:Literal['py', 'js']='py'  # Whether `onclick` is Python or JavaScript
):
    """Add a custom button to the solveit toolbar inside the shared button group."""
    _ensure_toolbar()
    icon = _resolve_icon(icon)

    if lang == 'js':
        iife(f"solvekit.addBtn({json.dumps(id)}, {json.dumps(icon)}, {onclick}, {json.dumps(tooltip)})")
    elif lang == 'py':
        js_onclick = dedent(f"""
        () => {{
            _post('upsert_msg', {{
                id_: '',
                msg_type: 'code',
                content: {json.dumps(onclick + '\nimport time; time.sleep(1)\nawait del_msg()')},
                is_input: '1',
                cmd: 'upsert-shift-run'
            }});
        }}""".strip())
        iife(f"solvekit.addBtn({json.dumps(id)}, {json.dumps(icon)}, {js_onclick}, {json.dumps(tooltip)})")

## Examples

**Dice Button**

Adds a toolbar button that rolls a random die. Click the button — it prints the result in a self-deleting code cell.

In [ ]:
add_btn('dice', 'ii:lucide:dice-5', 'import random; dice_num = random.randint(1,6); print(f"🎲 You rolled a {dice_num}!"); time.sleep(dice_num)', 'Roll a dice!', lang='py')

**Fastcore Docs Format Button**

Adds a toolbar button that reformats selected code cells into `fastcore docs format`. Select one or more code messages, then click the button — it creates a prompt asking SolveitAI to add docstrings and per-parameter comments to each function.

In [ ]:
fc_format = """
() => {
    const ids = selectedMsgIds();
    _post('upsert_msg', {
        id_: '',
        msg_type: 'prompt',
        content: `Please create a new version of functions in msgs ${ids} which adds a docstring and also adds a comment after each parameter. This is known as fastcore docs format. So each parameter will need to be on a separate line. Try to make the overall structure look the same as the previous function with these characteristics.`,
        is_input: '1',
        cmd: 'upsert-shift-run'
    });
}""".strip()

In [ ]:
add_btn('fc_format', 'ii:lucide:file-code', fc_format, 'Fastcore docs format', lang='js')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()